# RBF Kernel Ridge Regression — Color Harmony

### Why RBF Kernel Ridge instead of a neural network?

With 200–500 labeled pairs, a small MLP (e.g. 12→8→1 ≈ 113 parameters) can memorize
noise instead of learning real patterns. Kernel ridge regression sidesteps this:

- **Nonlinear** — the RBF kernel maps inputs into an infinite-dimensional space,
  capturing interactions that a polynomial or linear model cannot.
- **Few hyperparameters** — only γ (kernel width) and α (regularization). No learning
  rate schedule, no architecture search, no batch size.
- **Closed-form solution** — no gradient descent, no random initialization, fully
  reproducible.
- **PHP-portable** — prediction is a weighted sum of RBF similarities to the training
  points: `ŷ = Σ αᵢ · exp(−γ ‖x − xᵢ‖²) + intercept`. Easy to implement in any
  language without a ML library.
- **Regularized by design** — α (ridge penalty) shrinks coefficients, reducing
  overfitting on small data better than early stopping or dropout.

In [1]:
import pandas as pd
import numpy as np
from sklearn.kernel_ridge import KernelRidge
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from scipy.stats import spearmanr
from itertools import product
import warnings
warnings.filterwarnings('ignore')

# --- Hyperparameters (edit these) ---
ALPHA_GRID  = [0.001, 0.005, 0.01, 0.05, 0.1, 1, 10]      # ridge regularization (lambda)
GAMMA_GRID  = [0.01, 0.05, 0.1, 0.5, 1, 2]   # RBF width — small = smooth, large = spiky
N_FOLDS           = 5
RANDOM_STATE      = 42
OUTLIER_THRESHOLD = 50                        # max |rating - prediction| before dropping
WEIGHT_NEUTRAL    = 3                         # weight for rows with no 2nd rating
HARMONY_SIGMA     = 15.0                      # degrees bandwidth for Gaussian color harmony curves


## 1 · Data loading & cleaning

In [2]:
df = pd.read_csv('prediction-app/data/ratings.csv')
print(f'Raw rows: {len(df)}')

# Keep rows where prediction is NaN OR the two raters agree within threshold
has_pred = df['prediction'].notna()
df = df[~has_pred | ((df['rating'] - df['prediction']).abs() <= OUTLIER_THRESHOLD)].copy()
has_pred = df['prediction'].notna()  # recompute after filter

# Target: average the two human ratings; keep raw rating when no 2nd rating
df['y'] = df['rating'].astype(float)
df.loc[has_pred, 'y'] = (df.loc[has_pred, 'rating'] + df.loc[has_pred, 'prediction']) / 2
df['y'] = df['y'] / 100  # normalize to [0, 1]

# Weight: linear scale of rater agreement (1-10)
# Small |rating - prediction| -> high weight, large diff -> low weight
diff = (df.loc[has_pred, 'rating'] - df.loc[has_pred, 'prediction']).abs()
d_min, d_max = diff.min(), diff.max()
df.loc[has_pred, 'weight'] = 10 - (diff - d_min) / (d_max - d_min) * 9
df.loc[~has_pred, 'weight'] = WEIGHT_NEUTRAL

print(f'After filtering: {len(df)} rows ({has_pred.sum()} with prediction, {(~has_pred).sum()} without)')
df[['rating', 'prediction', 'y', 'weight']].describe()

Raw rows: 1257
After filtering: 1187 rows (428 with prediction, 759 without)


,rating,prediction,y,weight
count,1187.00000,428.000000,1187.000000,1187.000000
mean,44.31171,51.151869,0.450396,4.230497
std,31.88997,26.361615,0.302621,2.209111
min,1.00000,1.000000,0.010000,1.000000
25%,15.00000,29.000000,0.175000,3.000000
50%,40.00000,54.000000,0.420000,3.000000
75%,70.00000,71.000000,0.690000,4.960000
max,100.00000,100.000000,1.000000,10.000000


## 2 · Symmetric feature engineering & color theory

Every feature must satisfy `f(A, B) == f(B, A)` so swapping color order never changes the prediction.

### Color theory harmonies (smooth scale)
Instead of hard binary cuts (e.g. `1 if distance == 180 else 0`), we use smooth Gaussian bell curves:
$$\text{harmony} = \exp\left(-\frac{(\Delta h - \theta_0)^2}{2\sigma^2}\right)$$
This smoothly decays from 1.0 (exact match with harmony angle $\theta_0$) to 0.0, avoiding boundary cliffs.

| Harmony | Center $\theta_0$ | Interpretation |
|---|---|---|
| **Analogous** | 0° | Distance around 0–30° (adjacent / monochromatic hues) |
| **Square** | 90° | Tetradic 90° right angles on color wheel |
| **Triadic** | 120° | Equilateral triangle on color wheel |
| **Split-complementary** | 150° | Around 150° (adjacent to complementary) |
| **Complementary** | 180° | Opposite hues on color wheel |

| Raw columns | Symmetric features | What it captures |
|---|---|---|
| sat_1, sat_2 | sat_avg, sat_diff, sat_prod | saturation level, contrast, interaction |
| bri_1, bri_2 | bri_avg, bri_diff, bri_prod | brightness level, contrast, interaction |
| hue_1, hue_2 | hue_sin_sum, hue_cos_sum | pair position on the color wheel |
| hue_distance | hue_similarity | global angular closeness |
| is_neutral_1, is_neutral_2 | neutral_count | total neutral items (both_neutral removed) |


In [3]:
# Canonical pair ID for GroupKFold (sort colors so (A,B) and (B,A) stay in the same fold)
c1 = df[['hue_1', 'sat_1', 'bri_1']].apply(tuple, axis=1)
c2 = df[['hue_2', 'sat_2', 'bri_2']].apply(tuple, axis=1)
df['pair_id'] = [str(sorted([a, b])) for a, b in zip(c1, c2)]

# --- Symmetric features ---
# Circular hue position (summed for symmetry)
rad1, rad2 = np.deg2rad(df['hue_1']), np.deg2rad(df['hue_2'])
df['hue_sin_sum'] = np.sin(rad1) + np.sin(rad2)
df['hue_cos_sum'] = np.cos(rad1) + np.cos(rad2)
df['hue_similarity'] = 1 - df['hue_distance'] / 180

# Smooth color harmony features: Gaussian curves centered on theoretical angles
HARMONY_TARGETS = {
    'harmony_analogous': 0.0,      # 0-30° distance
    'harmony_square': 90.0,        # 90°
    'harmony_triadic': 120.0,      # 120°
    'harmony_split_comp': 150.0,   # 150°
    'harmony_complementary': 180.0 # 180°
}
for col_name, target_angle in HARMONY_TARGETS.items():
    df[col_name] = np.exp(- ((df['hue_distance'] - target_angle) ** 2) / (2 * HARMONY_SIGMA ** 2))

# Saturation & brightness: avg (level), diff (contrast), prod (interaction)
df['sat_avg']  = (df['sat_1'] + df['sat_2']) / 2
df['sat_prod'] = df['sat_1'] * df['sat_2']
df['bri_avg']  = (df['bri_1'] + df['bri_2']) / 2
df['bri_prod'] = df['bri_1'] * df['bri_2']
# sat_diff and bri_diff are already symmetric

# Neutral items (count only — both_neutral removed)
df['neutral_count'] = df['is_neutral_1'] + df['is_neutral_2']

FEATURES = [
    'hue_similarity', 'hue_sin_sum', 'hue_cos_sum',
    'harmony_analogous', 'harmony_square', 'harmony_triadic',
    'harmony_split_comp', 'harmony_complementary',
    'sat_avg', 'sat_diff', 'sat_prod',
    'bri_avg', 'bri_diff', 'bri_prod',
    'neutral_count',
]

X      = df[FEATURES].values
y      = df['y'].values
w      = df['weight'].values
groups = pd.Categorical(df['pair_id']).codes

# Center target: mean is exported as the model intercept
y_mean = y.mean()
y_c    = y - y_mean

print(f'{len(FEATURES)} symmetric features: {FEATURES}')
print(f'y_mean (intercept): {y_mean:.4f}')
print(f'Samples: {len(y)}, unique pairs: {len(set(groups))}')


15 symmetric features: ['hue_similarity', 'hue_sin_sum', 'hue_cos_sum', 'harmony_analogous', 'harmony_square', 'harmony_triadic', 'harmony_split_comp', 'harmony_complementary', 'sat_avg', 'sat_diff', 'sat_prod', 'bri_avg', 'bri_diff', 'bri_prod', 'neutral_count']
y_mean (intercept): 0.4504
Samples: 1187, unique pairs: 1186


## 3 · RBF Kernel Ridge — grid search (5-fold GroupKFold)

In [4]:
gkf   = GroupKFold(n_splits=N_FOLDS)
folds = list(gkf.split(X, y, groups))  # fix folds once for both models

best_mae    = np.inf
best_params = None
best_oof    = None

total_combos = len(ALPHA_GRID) * len(GAMMA_GRID)
print(f'Starting grid search: {total_combos} combinations ({N_FOLDS} folds = {total_combos * N_FOLDS} fits)...')

# Grid search over regularization alpha and RBF width gamma
for idx, (alpha, gamma) in enumerate(product(ALPHA_GRID, GAMMA_GRID), 1):
    oof = np.zeros(len(y))
    for train_idx, test_idx in folds:
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X[train_idx])
        X_te = scaler.transform(X[test_idx])

        kr = KernelRidge(kernel='rbf', alpha=alpha, gamma=gamma)
        kr.fit(X_tr, y_c[train_idx], sample_weight=w[train_idx])
        oof[test_idx] = kr.predict(X_te)

    mae = np.mean(np.abs(y_c - oof))
    new_best = False
    if mae < best_mae:
        best_mae    = mae
        best_params = {'alpha': alpha, 'gamma': gamma}
        best_oof    = oof.copy()
        new_best = True

    # Progress heartbeat: live ticker in-place; new line on improvement or checkpoint
    status = f'[{idx:>3}/{total_combos}] alpha={alpha:.4g}, gamma={gamma:.4g} -> MAE: {mae*100:.2f}p (Best: {best_mae*100:.2f}p)'
    if new_best or idx == 1 or idx == total_combos or idx % 25 == 0:
        print(status)
    else:
        print(f'\r{status}', end='', flush=True)

print()  # newline after live ticker

# Calculate Train R² across folds with best parameters
rbf_train_r2_scores = []
for train_idx, test_idx in folds:
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X[train_idx])
    kr = KernelRidge(kernel='rbf', alpha=best_params['alpha'], gamma=best_params['gamma'])
    kr.fit(X_tr, y_c[train_idx], sample_weight=w[train_idx])
    rbf_train_r2_scores.append(r2_score(y_c[train_idx], kr.predict(X_tr)))

rbf_train_r2 = np.mean(rbf_train_r2_scores)
rbf_test_r2  = r2_score(y, best_oof + y_mean)
rbf_rmse     = np.sqrt(np.mean((y_c - best_oof) ** 2))
rbf_mae      = best_mae
rbf_spearman = spearmanr(y, best_oof + y_mean).statistic

print(f'Best params:  alpha={best_params["alpha"]:.5f}, gamma={best_params["gamma"]:.5f}')
print(f'Train R²:     {rbf_train_r2:.4f}')
print(f'Test R²:      {rbf_test_r2:.4f}')
print(f'RMSE (MRSE):  {rbf_rmse:.4f}  ({rbf_rmse * 100:.1f} rating points)')
print(f'MAE:          {rbf_mae:.4f}  ({rbf_mae * 100:.1f} rating points)')
print(f'Spearman:     {rbf_spearman:.4f}')


Starting grid search: 42 combinations (5 folds = 210 fits)...
[  1/42] alpha=0.001, gamma=0.01 -> MAE: 26.28p (Best: 26.28p)
[  3/42] alpha=0.001, gamma=0.1 -> MAE: 34.91p (Best: 26.28p))[  4/42] alpha=0.001, gamma=0.5 -> MAE: 25.08p (Best: 25.08p)
[  6/42] alpha=0.001, gamma=2 -> MAE: 26.51p (Best: 25.08p)[  7/42] alpha=0.005, gamma=0.01 -> MAE: 24.31p (Best: 24.31p)
[ 12/42] alpha=0.005, gamma=2 -> MAE: 26.51p (Best: 24.31p)p))[ 13/42] alpha=0.01, gamma=0.01 -> MAE: 23.66p (Best: 23.66p)
[ 18/42] alpha=0.01, gamma=2 -> MAE: 26.51p (Best: 23.66p)p))[ 19/42] alpha=0.05, gamma=0.01 -> MAE: 22.61p (Best: 22.61p)
[ 24/42] alpha=0.05, gamma=2 -> MAE: 26.51p (Best: 22.61p)p))[ 25/42] alpha=0.1, gamma=0.01 -> MAE: 22.39p (Best: 22.39p)
[ 30/42] alpha=0.1, gamma=2 -> MAE: 26.51p (Best: 22.39p)p))[ 31/42] alpha=1, gamma=0.01 -> MAE: 22.15p (Best: 22.15p)
[ 41/42] alpha=10, gamma=1 -> MAE: 26.18p (Best: 22.15p)p))[ 42/42] alpha=10, gamma=2 -> MAE: 26.60p (Best: 22.15p)

Best params:  alpha=1.00

## 4 · MLP benchmark (same folds)

Architecture: `Features → 8 → 1`, Adam optimizer, L2 regularization (alpha=0.01), early stopping.  
Note: `MLPRegressor` does not support `sample_weight`, so it trains unweighted.


In [5]:
mlp_oof = np.zeros(len(y))
mlp_train_r2_scores = []

for train_idx, test_idx in folds:
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X[train_idx])
    X_te = scaler.transform(X[test_idx])

    mlp = MLPRegressor(
        hidden_layer_sizes=(8,),
        activation='relu',
        solver='adam',
        alpha=0.01,
        max_iter=2000,
        early_stopping=True,
        validation_fraction=0.15,
        random_state=RANDOM_STATE,
    )
    mlp.fit(X_tr, y_c[train_idx])
    mlp_train_r2_scores.append(r2_score(y_c[train_idx], mlp.predict(X_tr)))
    mlp_oof[test_idx] = mlp.predict(X_te)

mlp_train_r2 = np.mean(mlp_train_r2_scores)
mlp_test_r2  = r2_score(y, mlp_oof + y_mean)
mlp_rmse     = np.sqrt(np.mean((y_c - mlp_oof) ** 2))
mlp_mae      = np.mean(np.abs(y_c - mlp_oof))
mlp_spearman = spearmanr(y, mlp_oof + y_mean).statistic

print(f'Train R²:     {mlp_train_r2:.4f}')
print(f'Test R²:      {mlp_test_r2:.4f}')
print(f'RMSE (MRSE):  {mlp_rmse:.4f}  ({mlp_rmse * 100:.1f} rating points)')
print(f'MAE:          {mlp_mae:.4f}  ({mlp_mae * 100:.1f} rating points)')
print(f'Spearman:     {mlp_spearman:.4f}')


Train R²:     0.0845
Test R²:      -0.0165
RMSE (MRSE):  0.3050  (30.5 rating points)
MAE:          0.2483  (24.8 rating points)
Spearman:     0.3079


## 5 · Comparison

In [6]:
print(f'{"Model":<18} {"Train R²":>10} {"Test R²":>10} {"RMSE (MRSE)":>12} {"MAE":>10} {"Spearman":>10}')
print('-' * 74)
print(f'{"RBF KernelRidge":<18} {rbf_train_r2:>10.4f} {rbf_test_r2:>10.4f} {rbf_rmse*100:>11.1f}p {rbf_mae*100:>9.1f}p {rbf_spearman:>10.4f}')
print(f'{f"MLP {X.shape[1]}→8→1":<18} {mlp_train_r2:>10.4f} {mlp_test_r2:>10.4f} {mlp_rmse*100:>11.1f}p {mlp_mae*100:>9.1f}p {mlp_spearman:>10.4f}')
print('-' * 74)

if rbf_mae < mlp_mae and rbf_spearman > mlp_spearman and rbf_test_r2 > mlp_test_r2:
    print('\n✅ RBF wins across all validation metrics.')
elif rbf_mae >= mlp_mae and rbf_spearman <= mlp_spearman and rbf_test_r2 <= mlp_test_r2:
    print('\n⚠️  MLP wins across validation metrics.')
else:
    print('\n⚡ Mixed results — check which metric matters more for your task.')


Model                Train R²    Test R²  RMSE (MRSE)        MAE   Spearman
--------------------------------------------------------------------------
RBF KernelRidge        0.2825     0.2077        26.9p      22.1p     0.4524
MLP 15→8→1             0.0845    -0.0165        30.5p      24.8p     0.3079
--------------------------------------------------------------------------

✅ RBF wins across all validation metrics.


## 6 · Export final model to JSON for PHP

Train the final `KernelRidge` model on 100% of the dataset using the optimal hyperparameters found during cross-validation, then export the model weights, support vectors, and scaler parameters to `app/src/algorithm/model.json`.


In [7]:
import json
import os

# Fit final model on 100% of data with best hyperparameters
final_scaler = StandardScaler()
X_scaled = final_scaler.fit_transform(X)

final_kr = KernelRidge(kernel='rbf', alpha=best_params['alpha'], gamma=best_params['gamma'])
final_kr.fit(X_scaled, y_c, sample_weight=w)

model_export = {
    'gamma': float(best_params['gamma']),
    'intercept': float(y_mean),
    'harmony_sigma': float(HARMONY_SIGMA),
    'feature_names': FEATURES,
    'scaler_mean': final_scaler.mean_.tolist(),
    'scaler_scale': final_scaler.scale_.tolist(),
    'dual_coef': final_kr.dual_coef_.tolist(),
    'X_fit': X_scaled.tolist(),
}

# Resolve path whether running from project root or tests/ directory
export_path = 'app/src/algorithm/model.json' if os.path.exists('app') else '../app/src/algorithm/model.json'

with open(export_path, 'w') as f:
    json.dump(model_export, f)

print(f'Exported RBF model to: {export_path}')
print(f'Support vectors:       {len(model_export["X_fit"])}')
print(f'Features:              {len(model_export["feature_names"])}')
print(f'Intercept:             {model_export["intercept"]:.4f}')
print(f'Gamma:                 {model_export["gamma"]:.5f}')
print(f'Alpha:                 {best_params["alpha"]:.5f}')


Exported RBF model to: ../app/src/algorithm/model.json
Support vectors:       1187
Features:              15
Intercept:             0.4504
Gamma:                 0.01000
Alpha:                 1.00000
